In [ ]:
from pathlib import Path
import warnings
from concurrent.futures import ThreadPoolExecutor

import numpy as np
from skimage.morphology import erosion, dilation
from skimage.segmentation import watershed
from skimage.io import imread, imsave
from scipy.ndimage import gaussian_gradient_magnitude

from snap_to_edge import snap_labels_to_edge
from calmutils.morphology.structuring_elements import hypersphere_centered

def imsave_nowarnings(file, img, **kwargs):
    # catch low contrast warning
    with warnings.catch_warnings():
        warnings.simplefilter('ignore', UserWarning)
        imsave(file, img, **kwargs)

def process_single_image(mask_path, raw_path, out_path, edge_filter_sigmas, radius_morphology):
    
    # load
    labels = imread(mask_path)
    img = imread(raw_path)

    # filter to enhance edges
    edge_img = gaussian_gradient_magnitude(img.astype(float), edge_filter_sigmas)

    # refine labels
    labels_snap = snap_labels_to_edge(labels, edge_img, radius_morphology)

    # save as int (so it will be recognized as label by napari), compress to save significant space
    imsave_nowarnings(out_path, labels_snap.astype(int), compression=5)

In [ ]:
in_path = '/data/agl_data/AndreasMaiser/NSD/23AM07-05'

mask_subdirectory = 'segmentation_nuclei1'
raw_subdirectory = 'tif'
out_subdirectory = 'segmentation_nuclei1_edgesnap'

edge_filter_sigmas = (2, 1, 1)
radius_morphology = 2

In [ ]:
# get all TIFF files in mask directory
mask_files = sorted((Path(in_path) / mask_subdirectory).glob('*.tif'))

# get raw files by dropping suffix from mask filenames
mask_suffix = '_cp_masks'
raw_files = [Path(in_path) / raw_subdirectory / mask_file.name.replace(mask_suffix, '') for mask_file in mask_files]

In [ ]:
if not (Path(in_path) / out_subdirectory).exists():
    (Path(in_path) / out_subdirectory).mkdir(parents=True)

with ThreadPoolExecutor() as tpe:
    futures = []

    # enqueue
    for mask_file, raw_file in zip(mask_files, raw_files):
        out_file = Path(in_path) / out_subdirectory / mask_file.name
        f = tpe.submit(process_single_image, mask_file, raw_file, out_file, edge_filter_sigmas, radius_morphology)
        futures.append(f)
        
    # notify upon completion
    for f, mask_file in zip(futures, mask_files):
        f.result()
        print(f'processed {str(mask_file)}.')